# IMX500 전용 모델 변환 (양자화) & 다운로드 — 노트북 2

이 노트북은 **`finetune_imx500_yolo_1_train.ipynb`(노트북 1)** 의 **8단계(결과 확인)까지 완료한 뒤** 실행합니다.
기존 원본 노트북의 **9~11단계(IMX500 변환 · 다운로드 · 라즈베리파이 탑재)** 에 해당합니다.

**진행 순서**
1. 노트북 1의 **8-2 단계**에서 다운로드한 `{저장시각}_imx500_pkg.zip` 패키지를 준비합니다.
2. 이 노트북에서 설치 → 패키지 업로드 → **모델 파일명 입력**(또는 자동 선택) → INT8 양자화 → 다운로드를 진행합니다.

**시작하기 전:** 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 (양자화 자체는 CPU로도 동작하나, 노트북 1과 런타임 설정을 통일합니다)


## 0. GPU 확인


In [ ]:
!nvidia-smi -L
print('위에 Tesla T4 같은 게 안 보이면 런타임 유형을 GPU로 바꾸세요!')

## 1. 설치 (protobuf 호환성 수정 포함)

양자화(IMX500 변환)에 필요한 패키지를 설치합니다. (약 5~10분 소요)

> 🔧 **기존 오류 수정 — `cannot import name 'runtime_version' from 'google.protobuf'`**
>
> 기존 노트북 1의 설치 과정에서 `protobuf`가 4.25.5로 내려가면서,
> Colab 기본 설치된 **tensorflow 2.20.0**(`protobuf>=5.28.0` 요구)과 충돌하여 양자화 단계에서 위 오류가 발생했습니다.
> 이 노트북에서는 모든 설치가 끝난 뒤 **`protobuf>=5.28.0,<6.0` 을 별도로 설치**하여
> tensorflow와의 호환성을 확보합니다. (grpcio-status 요구사항 `protobuf<6.0dev`도 함께 만족)

설치 과정에서 Colab 기본 패키지(google-cloud-* 등)와의 일부 충돌 경고가 출력될 수 있으나,
이 노트북의 작업 흐름에는 영향이 없으므로 **무시하셔도 됩니다.**
셀 실행이 끝나면 런타임이 **자동으로 재시작**됩니다. "세션이 다운되었습니다" 메시지는 정상적인 현상이며, 재시작 완료 후 다음 셀부터 이어서 실행하세요.


In [ ]:
!apt-get -qq install -y openjdk-21-jre > /dev/null

!pip install -q ultralytics \
    "model-compression-toolkit>=2.4.1" "edge-mdt-cl<1.1.0" "edge-mdt-tpc>=1.2.0" \
    "pydantic<2.12" "imx500-converter[pt]>=3.17.3"

# tensorflow 2.20 호환 (runtime_version 오류 해결): protobuf 를 5.28 이상, 6.0 미만으로 고정
!pip install -q "protobuf>=5.28.0,<6.0"

# 교체된 패키지를 반영하려면 런타임을 한 번 재시작해야 합니다.
import os
os.kill(os.getpid(), 9)

런타임 재시작이 완료되면 아래 셀을 실행하여 설치 상태를 확인한 후 진행해 주세요.


In [ ]:
import ultralytics
ultralytics.checks()

## 2. 모델·데이터 준비

노트북 1의 **8-2 단계**에서 다운로드한 `{저장시각}_imx500_pkg.zip` 파일을 업로드합니다.

> **패키지 구성**
> - `{저장시각}.pt` : 양자화할 학습 모델
> - `data.yaml` : 클래스 정보 및 데이터 경로 설정
> - `config.json` : 학습 시 사용한 `IMGSZ`, 클래스명, 저장시각 (자동 복원)
> - `val/` : INT8 보정(calibration)에 사용할 실사진 검증 이미지 + 라벨


In [ ]:
from google.colab import files

print('노트북 1(8-2 단계)에서 받은 `{저장시각}_imx500_pkg.zip` 을 업로드하세요.')
uploaded = files.upload()

### 2-1. 업로드 패키지 자동 해제 및 설정

- 업로드한 zip 파일을 자동으로 찾아 `/content/imx500_pkg` 에 압축을 해제합니다.
- `data.yaml` 의 `path` 를 압축 해제 위치로 **자동 재설정**합니다.
- `config.json` 이 있으면 학습 당시의 **저장시각, `IMGSZ`, 클래스명** 을 자동 복원합니다.


In [ ]:
import os, glob, yaml, json, zipfile, shutil

assert 'uploaded' in globals() and uploaded, '업로드 셀을 먼저 실행하세요.'

# 업로드된 zip 자동 탐지
zip_files = [n for n in uploaded if n.lower().endswith('.zip')]
assert zip_files, '업로드한 파일에 .zip 이 없습니다. `{저장시각}_imx500_pkg.zip` 을 업로드하세요.'
PKG_ZIP = zip_files[0]
print('업로드된 패키지:', PKG_ZIP)

# 압축 해제
PKG_DIR = '/content/imx500_pkg'
shutil.rmtree(PKG_DIR, ignore_errors=True)
with zipfile.ZipFile(PKG_ZIP, 'r') as zf:
    zf.extractall(PKG_DIR)

# data.yaml 자동 탐지 + path 재설정
yaml_candidates = glob.glob(PKG_DIR + '/**/data.yaml', recursive=True)
assert yaml_candidates, 'data.yaml 을 찾지 못했습니다. 8-2 단계에서 만든 패키지인지 확인하세요.'
DATA_YAML = yaml_candidates[0]
cfg = yaml.safe_load(open(DATA_YAML))
cfg['path'] = PKG_DIR
yaml.safe_dump(cfg, open(DATA_YAML, 'w'), allow_unicode=True, sort_keys=False)
print('data.yaml path 재설정 ->', DATA_YAML)

# config.json 자동 복원 (없으면 기본값 사용)
AUTO_STAMP, AUTO_IMGSZ, AUTO_CLASS = None, 320, None
config_candidates = glob.glob(PKG_DIR + '/config.json')
if config_candidates:
    meta = json.load(open(config_candidates[0]))
    AUTO_STAMP = meta.get('stamp')
    AUTO_IMGSZ = meta.get('imgsz', 320)
    AUTO_CLASS = meta.get('class_name')
    print(f'config.json 복원 -> 저장시각: {AUTO_STAMP}, IMGSZ: {AUTO_IMGSZ}, 클래스: {AUTO_CLASS}')

# .pt 모델 자동 탐지 후보
PT_CANDIDATES = sorted(set(glob.glob(PKG_DIR + '/*.pt') + glob.glob(PKG_DIR + '/**/*.pt', recursive=True)))
print('패키지 내 .pt 모델 목록:', [os.path.basename(p) for p in PT_CANDIDATES])
assert PT_CANDIDATES, '.pt 모델 파일을 찾지 못했습니다. 8-2 단계에서 만든 패키지인지 확인하세요.'

### 2-2. 모델 파일명 입력

노트북 1에서 저장한 모델 파일명(`저장시각시분초.pt`)을 아래에 입력하세요.
**비워 두면** 업로드된 패키지에서 `.pt` 파일을 **자동으로 선택**합니다.

> 패키지에 포함된 모델과 다른 파일을 양자화하려면, 그 파일을 `/content/imx500_pkg/` 안에 넣은 뒤 파일명을 입력하면 됩니다.


In [ ]:
# =====================================================================
# 이전 노트북(1)에서 저장한 모델 파일명을 입력하세요.
# 예: MODEL_NAME = '20260815_145030.pt'
# 비워 두면 업로드된 패키지의 .pt 를 자동 선택합니다.
MODEL_NAME = ''
# =====================================================================

In [ ]:
# 입력값 검증 및 자동 선택
if MODEL_NAME.strip():
    if not MODEL_NAME.endswith('.pt'):
        MODEL_NAME += '.pt'
    assert os.path.exists(f'{PKG_DIR}/{MODEL_NAME}'), f'{MODEL_NAME} 파일이 패키지에 없습니다. 2-2 셀의 파일명을 확인하세요.'
    print('선택한 모델:', MODEL_NAME)
else:
    MODEL_NAME = os.path.basename(PT_CANDIDATES[-1])
    print('자동 선택된 모델:', MODEL_NAME)

MODEL_PT = f'{PKG_DIR}/{MODEL_NAME}'

# 저장시각 / IMGSZ 결정 (config.json 우선, 없으면 기본값)
STAMP = os.path.splitext(MODEL_NAME)[0]
IMGSZ = AUTO_IMGSZ
if AUTO_STAMP:
    STAMP = AUTO_STAMP

# 보정(calibration)용 검증 이미지 확인
val_imgs = sorted(glob.glob(f'{PKG_DIR}/val/images/*'))
assert len(val_imgs) > 0, '보정(calibration)용 이미지(val/images) 를 찾지 못했습니다. 패키지를 확인하세요.'

print()
print('모델 파일   :', MODEL_PT)
print('data.yaml   :', DATA_YAML)
print('IMGSZ       :', IMGSZ)
print('저장시각     :', STAMP)
print('보정용 이미지:', len(val_imgs), '장')
if len(val_imgs) < 300:
    print('⚠️ 300장 미만이라 INT8 보정 경고가 표시될 수 있지만 정상 동작합니다. 필요 시 `val/images` 에 보정용 이미지를 추가하세요.')

## 9. IMX500 전용 모델 변환 (양자화)

IMX500 하드웨어 칩셋은 부동소수점 연산을 지원하지 않으므로 모델을 **8비트 정수형으로 압축**합니다.
`data=` 매개변수로 지정된 이미지 데이터를 기반으로 자동 보정(calibration)이 수행됩니다.

변환 작업은 약 **5~10분** 소요되며, 진행되는 동안 런타임을 중단하지 마세요.

> `Exporting on CPU while CUDA is available...` 경고 메시지는 정상적인 출력 현상입니다.


In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_PT)
export_dir = model.export(format='imx', data=DATA_YAML, imgsz=IMGSZ)
print('\n변환 결과 폴더:', export_dir)

!ls -la {export_dir}

## 10. 다운로드

변환 결과물(`packerOut.zip`, `labels.txt`)과 원본 모델 백업을 `{저장시각}_imx500.zip` 으로 묶어 다운로드합니다.


In [ ]:
import shutil, os
from google.colab import files

OUT = '/content/imx500_out'
shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(OUT, exist_ok=True)

shutil.copy(os.path.join(export_dir, 'packerOut.zip'), OUT)
shutil.copy(os.path.join(export_dir, 'labels.txt'), OUT)
shutil.copy(MODEL_PT, os.path.join(OUT, 'best.pt'))          # 재학습용 백업

print('클래스 목록 (config.yaml 의 target_class 에 이 이름을 씁니다):')
print(open(os.path.join(OUT, 'labels.txt')).read())

OUT_ZIP = f'/content/{STAMP}_imx500.zip'
shutil.make_archive(f'/content/{STAMP}_imx500', 'zip', OUT)
print('다운로드 파일:', OUT_ZIP)
files.download(OUT_ZIP)

## 11. 라즈베리파이 탑재

05-custom-model 문서의 4. 라즈베리파이 탑재 및 모델 설정 을 참고해 라즈베리파이에 모델을 탑재합니다.


---
## 트러블슈팅 (IMX500 변환 관련)

| 증상 | 해결 방법 |
|---|---|
| `cannot import name 'runtime_version' from 'google.protobuf'` 오류 | protobuf 가 4.x 로 낮아진 상태입니다. 런타임을 재시작한 뒤 1단계 설치 셀(`protobuf>=5.28.0,<6.0` 포함)부터 다시 실행하세요. |
| export 단계에서 기타 오류 발생 | 런타임을 재시작한 후 설치 확인 셀부터 다시 순차 실행하세요. |
| `>300 images recommended for INT8 calibration` 경고 | 보정 이미지가 300장 미만일 때 표시되는 경고입니다. 정상 동작하나 정확도가 떨어질 수 있습니다. `val/images` 에 보정용 이미지를 추가한 뒤 다시 변환하세요. |
| AI 카메라 탑재 시에만 성능 저하 | 양자화에 따른 정밀도 손실 현상입니다. 노트북 1에서 `IMGSZ = 640`으로 설정 후 5-2 단계부터 재학습하세요. |
